In [22]:
import ROOT as root
from ROOT import TFile, TTree, TList
import numpy as np
import matplotlib.pyplot as plt
import glob

In [23]:
nC=1e9
nofEvents = 50000000    
fScaleFactorMU = 1/2.3514811546434146e-13
scale_factor = (1./nofEvents)*fScaleFactorMU

In [ ]:
TotalCharge1 = []
TotalCharge2 = []

AnglesArray = [0,10,20,30,40,50,60,70,80,90]
#AnglesArray = ['2x2cm','4x4cm','6x6cm','8x8cm','10x10cm','12x12cm','14x14cm','16x16cm']

SensorReadout1 = np.zeros([32,6,np.size(AnglesArray)])
SensorReadout2 = np.zeros([32,6,np.size(AnglesArray)])

k=0
for angle in AnglesArray:
     print(str(angle),"deg data processing ...")
     pathList = glob.glob("BB7/20240926_083620/"+ str(angle) +"/*/BB7Readout.root")

     j=0
     for path in pathList:
          print("File",str(j),"is being read ...")
          f = root.TFile(path)
          myTree = f.Get("BB7Hits")
          entries = myTree.GetEntriesFast()

          for entry in myTree:         
               # Now you have acess to the leaves/branches of each entry in the tree, e.g.
               if(entry.SensorID==0):
                    TotalCharge1.append(entry.Edep)
                    # conversion to int
                    i = int(entry.StripID)
                    SensorReadout1[i,j,k] += entry.Edep
               if(entry.SensorID==1):
                    TotalCharge2.append(entry.Edep)
                    i = int(entry.StripID)
                    SensorReadout2[i,j,k] += entry.Edep
          j=j+1
     k=k+1

In [ ]:
fig=plt.figure(figsize=(6,3))

x=np.linspace(0, 31, 32)

ax1=fig.add_subplot(121)
ax1.set_ylim(0,0.05)
ax1.set_title('Sensor 0', fontsize=10)
ax1.set_xlabel('StripID', fontsize=8,labelpad=8)
ax1.set_ylabel('Total charge [C/UM]', fontsize=8,labelpad=8)
ax1.grid(linestyle='--', linewidth=0.25)

k=0
for angle in AnglesArray:
    ax1.errorbar(x,np.mean(SensorReadout1[:,:,k],1)*scale_factor/nC,yerr=np.std(SensorReadout1[:,:,k],1)*scale_factor/nC, ds='steps-mid', label=str(angle) + ' deg')
    k=k+1


ax1.legend(loc='best', fontsize="6")

ax2=fig.add_subplot(122)
ax2.set_ylim(0,0.05)
ax2.set_title('Sensor 1', fontsize=10)
ax2.set_xlabel('StripID', fontsize=8,labelpad=8)
ax2.set_ylabel('Total charge [C/UM]', fontsize=8,labelpad=8)
ax2.grid(linestyle='--', linewidth=0.25)

k=0
for angle in AnglesArray:
    ax2.errorbar(x,np.mean(SensorReadout2[:,:,k],1)*scale_factor/nC,yerr=np.std(SensorReadout2[:,:,k],1)*scale_factor/nC, ds='steps-mid', label=str(angle) + ' deg')
    k=k+1

ax2.legend(loc='best', fontsize="6")

fig.tight_layout()
plt.show()